In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import default_data_collator
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import numpy as np
import pandas as pd
from datasets import Dataset 
import torch
import re
import tensorflow as tf
import numpy as np
import re
from llm_to_change_real_to_fake import generate_fake_news

/media/tanmoy002/HDD/fake_real_news_pipeline/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-21 12:32:29.487871: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-21 12:32:29.532979: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-21 12:32:54.556406: I tensorflow/core/util/port.cc:153] oneDNN custom operations 

In [2]:
# data=pd.read_excel("web_news.xlsx")

In [3]:
# # Check column names
# print("Columns in Excel:", data.columns.tolist())

# # Select 2nd and 4th columns (indexing starts at 0)
# selected_df = data.iloc[:, [2, 4]]  # 2nd column = index 1, 4th column = index 3

# # Optional: rename columns for clarity
# selected_df.columns = ['title', 'description']

# # Save to CSV
# selected_df.to_csv("web_news_selected.csv", index=False)

# print("Saved successfully as web_news_selected.csv")

In [2]:
data_ = pd.read_csv("web_news_selected_.csv", encoding="utf-8")
# Add a new column 'label' with all values 0
data_['label'] = 1


In [3]:
data_

,title,description,label
0,কুষ্টিয়ার এসপির নামে পুরনো ও ভুয়া নাচের ভিডি...,"“দেখুন কুষ্টিয়ার এসপি কি করছে, নাউজুবিল্লাহ, ন...",1
1,মিজানুর রহমান আজহারিকে চাঁদে দেখতে পাওয়ার গুজব,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুকের একটি...",1
2,বদি ভাই খ্যাত অভিনেতা আব্দুল কাদেরের মৃত্যুর গুজব,ফেসবুকের বিভিন্ন আইডি ও পেজের মাধ্যমে গত ১৯ ডি...,1
3,সেনাবাহিনীর প্রথম নারী মেজর জেনারেলের ছবি বিকৃ...,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুক ও ইউটি...",1
4,হ্যাকার হামজার পরিবর্তে ভিন্ন ছবি ব্যবহার করে ...,বিগত কয়েকবছর ধরে ফেসবুক এবং বিভিন্ন দেশী ভুঁই...,1
...,...,...,...
22062,অনুরূপ সেশনগুলিতে নূর ভাইকে একটি চিঠিও দেখা যাবে,"অভিনেতা আসাদুজ্জামান নূর, বাংলাদেশি কথাচিত্র প...",1
22063,Thai Parliament Ignites Diplomatic Crisis by D...,"In a surprise move, the Thai government has sp...",1
22064,Trump Lawyers Unveil Groundbreaking Proposal t...,"In a major development, President Trump's lega...",1
22065,Moscow Concert Hall Siege Ends with Unexpected...,"In a shocking turn of events, authorities reve...",1


In [4]:
# 1. Drop rows where the 'description' column is NaN
df_clean = data_.dropna(subset=['title', 'description'])
df_clean = df_clean[(df_clean['title'].str.strip() != '') & (df_clean['description'].str.strip() != '')]
# 2. Optional: reset index
df_clean = df_clean.reset_index(drop=True)

In [5]:
df_clean

,title,description,label
0,কুষ্টিয়ার এসপির নামে পুরনো ও ভুয়া নাচের ভিডি...,"“দেখুন কুষ্টিয়ার এসপি কি করছে, নাউজুবিল্লাহ, ন...",1
1,মিজানুর রহমান আজহারিকে চাঁদে দেখতে পাওয়ার গুজব,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুকের একটি...",1
2,বদি ভাই খ্যাত অভিনেতা আব্দুল কাদেরের মৃত্যুর গুজব,ফেসবুকের বিভিন্ন আইডি ও পেজের মাধ্যমে গত ১৯ ডি...,1
3,সেনাবাহিনীর প্রথম নারী মেজর জেনারেলের ছবি বিকৃ...,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুক ও ইউটি...",1
4,হ্যাকার হামজার পরিবর্তে ভিন্ন ছবি ব্যবহার করে ...,বিগত কয়েকবছর ধরে ফেসবুক এবং বিভিন্ন দেশী ভুঁই...,1
...,...,...,...
22062,অনুরূপ সেশনগুলিতে নূর ভাইকে একটি চিঠিও দেখা যাবে,"অভিনেতা আসাদুজ্জামান নূর, বাংলাদেশি কথাচিত্র প...",1
22063,Thai Parliament Ignites Diplomatic Crisis by D...,"In a surprise move, the Thai government has sp...",1
22064,Trump Lawyers Unveil Groundbreaking Proposal t...,"In a major development, President Trump's lega...",1
22065,Moscow Concert Hall Siege Ends with Unexpected...,"In a shocking turn of events, authorities reve...",1


In [6]:
data_real=pd.read_csv("real_news.csv")
data_real['label']=0

In [7]:
data_real

,id,ordering,datetime,agency,news_title,details_link,details_content,sentiment,summary,category_id,type,status,created_at,updated_at,district,thana,crime_name,processed,label
0,284,0,2024-01-30 03:12:40,Dawn,"TTP, IS-K, BLA behind 82pc deaths in terror at...",https://www.dawn.com/news/1803010/ttp-is-k-bla...,ISLAMABAD: Banned groups like the Tehreek-i-Ta...,"negative (polarity=-0.06416666666666665, subje...","Of these strikes, 97 occurred in KP, 28 in Bal...",3,web,1,2024-01-30 03:12:41,2024-01-30 03:12:41,NaN,NaN,"violence, killed",1,0
1,402,0,2024-01-30 09:52:37,Ananda Bazar,Terms of Use,https://www.anandabazar.com/termsofuse,These Terms of Use govern your use of the webs...,"positive (polarity=0.048377976190476166, subje...",These Terms of Use govern your use of the webs...,3,web,1,2024-01-30 09:52:38,2024-01-30 09:52:38,NaN,NaN,"Theft, Fraud, Perjury, violation",1,0
2,403,0,2024-01-30 09:52:37,Ananda Bazar,About Us,https://www.anandabazar.com/aboutus,আনন্দবাজার পত্রিকা যাত্রা শুরু করেছিল ১৩ মার্চ...,"positive (polarity=0.035695947570947566, subje...","আনন্দ বাজার ম্যাগাজিনটি March ই মার্চ, 1222 এ ...",3,web,1,2024-01-30 09:52:38,2024-01-30 09:52:38,NaN,NaN,NaN,1,0
3,404,0,2024-01-30 09:52:37,Ananda Bazar,Privacy Policy,https://www.anandabazar.com/privacy,"Last Updated on July 07, 2021 This Privacy Pol...","positive (polarity=0.04553719497749349, subjec...",The information gathered when you interact wit...,3,web,1,2024-01-30 09:52:38,2024-01-30 09:52:38,NaN,NaN,NaN,1,0
4,449,0,2024-01-30 09:56:05,The Washington Post,Tracking Biden administration political appoin...,https://www.washingtonpost.com/politics/intera...,The Washington Post is providing this importan...,"positive (polarity=0.09407547109752992, subjec...",The Washington Post and thePartnership for Pub...,2,web,1,2024-01-30 09:56:06,2024-01-30 09:56:06,NaN,NaN,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,201453,0,2024-03-28 19:07:36,Dainik Sylhet,ডেইলি সিলেট ডেস্ক :: বিয়ের প্রলোভনে প্রেমের সম...,https://dailysylhet.com/details/722589/,cialis fiyatcialis siparişhttp://umraniyetip...,"positive (polarity=0.06465909090909092, subjec...",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,3,web,1,2024-03-28 19:07:36,2024-03-28 19:07:36,সুনামগঞ্জ,NaN,NaN,1,0
49996,201454,0,2024-03-28 19:07:36,Dainik Sylhet,ডেইলি সিলেট ডেস্ক :: পিংকু বিশ্বাস মৌলভীবাজারে...,https://dailysylhet.com/details/722586/,cialis fiyatcialis siparişhttp://umraniyetip...,"positive (polarity=0.047568118156353464, subje...",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,3,web,1,2024-03-28 19:07:36,2024-03-28 19:07:36,মৌলভীবাজার,NaN,NaN,1,0
49997,201456,0,2024-03-28 19:07:36,Dainik Sylhet,ডেইলি সিলেট ডেস্ক :: যুক্তরাষ্ট্র প্রবাসী বাংল...,https://dailysylhet.com/details/722577/,cialis fiyatcialis siparişhttp://umraniyetip...,"positive (polarity=0.18938878581735724, subjec...",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিপি: //umr...,3,web,1,2024-03-28 19:07:36,2024-03-28 19:07:36,সিলেট,NaN,হত্যা,1,0
49998,201457,0,2024-03-28 19:07:36,Dainik Sylhet,"দেশে কোটিপতি হিসাবধারীর সংখ্যা বেড়েছে ৬,৯৬২টি",https://dailysylhet.com/details/721989/,cialis fiyatcialis siparişhttp://umraniyetip...,"positive (polarity=0.06118561710398445, subjec...",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,3,web,1,2024-03-28 19:07:36,2024-03-28 19:07:36,সিলেট,NaN,পাচার,1,0


In [8]:
import pandas as pd

# Select 2nd, 4th, and 18th columns
selected_df_ = data_real.iloc[:, [4, 8, 18]]  

# Rename for clarity
selected_df_.columns = ['title', 'details_count', 'label']

# Drop rows where title and summary are exactly the same
# selected_df_ = selected_df_.drop_duplicates(subset=['title', 'details_count'], keep='first')
selected_df_.rename(columns={'details_count': 'description'}, inplace=True)
selected_df_ = selected_df_.dropna(subset=['title', 'description'])
selected_df_ = selected_df_[(selected_df_['title'].str.strip() != '') & (selected_df_['description'].str.strip() != '')]
# Optional: reset index
selected_df_ = selected_df_.reset_index(drop=True)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              


/tmp/ipykernel_17539/539936309.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df_.rename(columns={'details_count': 'description'}, inplace=True)


In [9]:
selected_df_

,title,description,label
0,"TTP, IS-K, BLA behind 82pc deaths in terror at...","Of these strikes, 97 occurred in KP, 28 in Bal...",0
1,Terms of Use,These Terms of Use govern your use of the webs...,0
2,About Us,"আনন্দ বাজার ম্যাগাজিনটি March ই মার্চ, 1222 এ ...",0
3,Privacy Policy,The information gathered when you interact wit...,0
4,Tracking Biden administration political appoin...,The Washington Post and thePartnership for Pub...,0
...,...,...,...
49995,ডেইলি সিলেট ডেস্ক :: বিয়ের প্রলোভনে প্রেমের সম...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0
49996,ডেইলি সিলেট ডেস্ক :: পিংকু বিশ্বাস মৌলভীবাজারে...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0
49997,ডেইলি সিলেট ডেস্ক :: যুক্তরাষ্ট্র প্রবাসী বাংল...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিপি: //umr...,0
49998,"দেশে কোটিপতি হিসাবধারীর সংখ্যা বেড়েছে ৬,৯৬২টি",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0


In [10]:
print(selected_df_.columns)


Index(['title', 'description', 'label'], dtype='object')


In [13]:
# selected_df_random = selected_df_.sample(45000, random_state=42)
# import time
# # Generate fake news with retries
# for idx, row in selected_df_random.iterrows():
#     fake_title, fake_description = generate_fake_news(row["title"], row["description"])
#     if fake_title == "N/A" or fake_description == "N/A":
#         print(f"Skipping row {idx} due to generation failure.")
#         continue
#     else:
#         print("\nOriginal Real News:")
#         print(row["title"], "\n", row["description"])
#         print("\nGenerated Fake News:")
#         print(fake_title, "\n", fake_description)
#         print("\n")
    
#     time.sleep(0.5)


In [11]:
# Merge the two dataframes
merged_df = pd.concat([df_clean, selected_df_], ignore_index=True)
# Convert 'title' and 'description' columns to lowercase
merged_df['title'] = merged_df['title'].str.lower()
merged_df['description'] = merged_df['description'].str.lower()


In [12]:
merged_df

,title,description,label
0,কুষ্টিয়ার এসপির নামে পুরনো ও ভুয়া নাচের ভিডি...,"“দেখুন কুষ্টিয়ার এসপি কি করছে, নাউজুবিল্লাহ, ন...",1
1,মিজানুর রহমান আজহারিকে চাঁদে দেখতে পাওয়ার গুজব,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুকের একটি...",1
2,বদি ভাই খ্যাত অভিনেতা আব্দুল কাদেরের মৃত্যুর গুজব,ফেসবুকের বিভিন্ন আইডি ও পেজের মাধ্যমে গত ১৯ ডি...,1
3,সেনাবাহিনীর প্রথম নারী মেজর জেনারেলের ছবি বিকৃ...,"সম্প্রতি, সামাজিক যোগাযোগ মাধ্যম ফেসবুক ও ইউটি...",1
4,হ্যাকার হামজার পরিবর্তে ভিন্ন ছবি ব্যবহার করে ...,বিগত কয়েকবছর ধরে ফেসবুক এবং বিভিন্ন দেশী ভুঁই...,1
...,...,...,...
72062,ডেইলি সিলেট ডেস্ক :: বিয়ের প্রলোভনে প্রেমের সম...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0
72063,ডেইলি সিলেট ডেস্ক :: পিংকু বিশ্বাস মৌলভীবাজারে...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0
72064,ডেইলি সিলেট ডেস্ক :: যুক্তরাষ্ট্র প্রবাসী বাংল...,সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিপি: //umr...,0
72065,"দেশে কোটিপতি হিসাবধারীর সংখ্যা বেড়েছে ৬,৯৬২টি",সিআইলিস ফাইয়েটসিয়ালিস সিপারিআইএইচটিটিপি: //u...,0


In [13]:
# Remove duplicates based on title and description
# merged_df_ = merged_df.drop_duplicates(subset=['title', 'description'], keep='first')
# Shuffle randomly
merged_df_ = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [14]:
merged_df_

,title,description,label
0,আজ বৃহত্তর জামাতে ইসলামের বিএনপি শাখার একটি প্...,বিজ্ঞপ্তীটি অবিক্রিত অবস্থায় প্রকাশ করা হল একট...,1
1,বাংলাদেশের প্রথম জুগন্তর পত্রিকার বারিসাল ব্যু...,জুগন্তর পত্রিকার বারিশাল ব্যুরো থেকে প্রতিদিনে...,1
2,আরটিজেএ নির্বাচনে লড়ছেন ১৫ প্রার্থী,"আখতার রহমান, ইএমএসএস, গণ যোগাযোগ ও সাংবাদিকতা,...",0
3,"ukraine secretly builds ""doomsday"" arsenal to ...","in a shocking revelation, high-ranking officia...",1
4,"পড়াশোনা আসলে চাকরি পাওয়ার জন্য নয়, জ্ঞান অর্জন...",উত্স: আজকের ম্যাগাজিন অনলাইন নিউজ: মোহানগঞ্জ উ...,0
...,...,...,...
72062,the icon interview: will of steel,"“in modelling, you’re expected to look glamoro...",0
72063,"চিট্টাগংয়ের হাথাজারীতে 10 জন নিহত, 20 জন আহত",সোমবার (29th november) দুপুর 12:00 pm-এ চিট্টা...,1
72064,বিস্তারিত,"হারুন, কাউন্সিলর জাফরুল হায়দার এবং অন্যান্য ও...",0
72065,আর্থিক প্রতারণার উদ্দেশ্যে ভারতীয় শিশু আরাসালা...,সম্প্রতি “আমিন না লিখে যাবেন না ” শীর্ষক শিরোন...,1


In [19]:
merged_df_.to_csv("final_dataset.csv", encoding="utf-8",index=False)

In [20]:
temp_data=pd.read_csv("final_dataset.csv")
temp_data.head

<bound method NDFrame.head of                                                    title  \
0      আজ বৃহত্তর জামাতে ইসলামের বিএনপি শাখার একটি প্...   
1      বাংলাদেশের প্রথম জুগন্তর পত্রিকার বারিসাল ব্যু...   
2                    আরটিজেএ নির্বাচনে লড়ছেন ১৫ প্রার্থী   
3      ukraine secretly builds "doomsday" arsenal to ...   
4      পড়াশোনা আসলে চাকরি পাওয়ার জন্য নয়, জ্ঞান অর্জন...   
...                                                  ...   
72062                  the icon interview: will of steel   
72063      চিট্টাগংয়ের হাথাজারীতে 10 জন নিহত, 20 জন আহত   
72064                                          বিস্তারিত   
72065  আর্থিক প্রতারণার উদ্দেশ্যে ভারতীয় শিশু আরাসালা...   
72066  রাজশাহীর গোদাগাড়ীতে সিপিএসসি, র‌্যাব-৫ অন্তর্ভ...   

                                             description  label  
0      বিজ্ঞপ্তীটি অবিক্রিত অবস্থায় প্রকাশ করা হল একট...      1  
1      জুগন্তর পত্রিকার বারিশাল ব্যুরো থেকে প্রতিদিনে...      1  
2      আখতার রহমান, ইএমএসএস, গণ যোগাযোগ ও সাংবাদিকত

In [ ]:
temp_data.columns

Index(['title', 'description', 'label'], dtype='object')

: 

In [18]:
# 1. Load the model
model_embedding = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [19]:
# 3. Combine title and summary for embeddings
texts = (merged_df_['title'] + ". " + merged_df_['description']).tolist()
labels = merged_df_['label'].tolist()  # 0 = Real, 1 = Fake

In [20]:
texts

['আজ বৃহত্তর জামাতে ইসলামের বিএনপি শাখার একটি প্রেস বিজ্ঞপ্তী মতিকণ্ঠে এসে পৌছেছে. বিজ্ঞপ্তীটি অবিক্রিত অবস্থায় প্রকাশ করা হল একটা ঘষোনাবৃহত্তর জামাতে ইসলামের বিএনপি শাখার ভবিষ্যত কান্ডারি তারেক জিয়া লন্ডনে দির্ঘ কাল বেপি চিকিৎস্বাধীন জীবন যাপন করছেন  কোমর ও মেরু দন্ড সারানোর পর এখন দু দন্ড সময় বেয় করে তিনি বাকশালী ভন্ড স্ট্রেটেজি পন্ড করে খন্ড খন্ড গন্ডগোল বাধিয়ে দিতে সক্ষম হয়েছেন  দেশ জুড়ে এখন চলছে মনির পোড়ানর উৎসব  এ উপলখ্যে তিনি একটি ভিডিও বার্তা প্রকাশ করেছেন ভিডিও তে তিনি বলেছেন, শান্তির জন্য শক্তি প্রয়গ করা প্রয়জন  এটা পৃথিবির ইতিহাসে হয়েছে, আমাদের ইসলাম ধর্মেও হয়েছে বিএনপি শাখার চলমান অবরধ আন্দলনকে শান্তিকামি আক্ষা দিয়ে এর সহিত ইসলামের আন্দলনের মিল খুজে পাওয়ার কৃতিত্তের স্বিকৃতি হিসাবে বিএনপি শাখার পক্ষ হতে একটি সন্মান সুচক উপাধি তার নামের শেষে ব্রেকেটে যোগ করার সিদ্ধান্ত নেওয়া হয়েছে এখন হতে তার পুর্ন নাম – তারেক জিয়া (খাঃ পুঃ), এখানে খাঃ পুঃ অর্থ খালেদার পুত্র  বস্তুত ছোট গনতন্ত্রের মৃত্যুর পর উপাধির দাবিটি তার একচ্ছত্র হয়েছে বিএনপি শাখার আরেকটি সিদ্ধান্ত অনুযায়ি গনতন্ত্রের ছো

In [21]:
labels

[1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,


In [ ]:
# 4. Convert texts to embeddings
embeddings = model_embedding.encode(texts)

In [ ]:
embeddings

array([[ 0.3351668 ,  0.25482088,  0.40661204, ..., -0.12484014,
         0.5485059 ,  0.05076706],
       [ 0.18050972,  0.20933747,  0.2968672 , ..., -0.07217188,
         0.42048368,  0.1247208 ],
       [-0.09412666,  0.20193774,  0.15622883, ...,  0.10037111,
         0.09395605,  0.05146353],
       ...,
       [ 0.12605944,  0.2360253 ,  0.2127403 , ..., -0.08981299,
         0.18606715, -0.02119192],
       [ 0.11086766,  0.12916143,  0.10594821, ...,  0.08316283,
         0.24745876,  0.14446992],
       [-0.1126496 ,  0.17012505, -0.07345963, ..., -0.09644586,
         0.15832955,  0.01071112]], shape=(72067, 384), dtype=float32)

In [ ]:
embeddings[0]

array([ 0.3351668 ,  0.25482088,  0.40661204, -0.04237692, -0.8459396 ,
       -0.24372058,  0.9841131 , -0.01160734,  0.14737672, -0.08282645,
        0.36888707, -0.6715539 ,  0.70983744, -0.15631653,  0.40338928,
       -0.11645518, -0.00576739,  0.3013404 , -0.37053537, -0.6189693 ,
       -0.00534362, -0.21765697, -0.15444049, -0.00265992, -0.23823033,
        0.10180818, -0.0326643 ,  0.10217959,  0.20038067,  0.6492315 ,
        0.27287215,  0.51601124,  0.1220592 ,  0.327273  ,  0.10172254,
        0.30640304, -0.3206563 ,  0.00209171, -0.02922114,  0.1706832 ,
       -0.45669037, -0.06885256,  0.45980197,  0.7255845 ,  0.03828955,
        0.27527013, -0.09543827, -0.39205825,  0.03246784, -0.503428  ,
        0.03763621, -0.3456446 , -0.46816975,  0.02673538,  0.02063367,
        0.18695568,  0.2184712 , -0.2832603 , -0.20589042,  0.06994922,
        0.00347422,  0.0752637 , -0.21147975,  0.14608347, -0.8182585 ,
        0.31609386,  0.17731448,  0.27716303,  0.02867348,  0.04

In [ ]:
len(embeddings[0])

384

In [ ]:
# 5. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

### Logistic Regression

In [ ]:
# 6. Train a classifier
clf = LogisticRegression()
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [ ]:
# 7. Predict and evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9384626057999167


In [ ]:
# 8. Predict new text
new_title = "Advisors Mahfuz Alam and Asif Mahmud resign today"
new_summary = "Two student representatives on the interim government's advisory council are set to resign on Wednesday. These two advisors are Information and Broadcasting Advisor Mahfuz Alam and Local Government Advisor Asif Mahmud Sajeeb Bhuiyan."
"This information was obtained from responsible government sources. However, no one wanted to speak on the matter anonymously."

"The Election Commission (EC) is likely to announce the schedule for the 13th parliamentary elections on Wednesday evening or Thursday. Sources in the government have said that two advisors will resign before that. The two advisors informed the government officials about the matter on Tuesday."

"Meanwhile, Advisor Asif Mahmud has called a press conference at the Secretariat today, Wednesday at 3 pm. A press release sent by Md. Salahuddin, Public Relations Officer of the Ministry of Local Government, said that the advisor will talk about contemporary issues."
new_text = [new_title + ". " + new_summary]
new_emb = model_embedding.encode(new_text)
prediction = clf.predict(new_emb)
print("Fake" if prediction[0]==1 else "Real")

Fake


In [ ]:
len(prediction)


1

### Finetunning bert transformer model for deteting fake news or real news

### Pretrained Model Knowledge
 - Models like bert-base-multilingual-uncased have been trained on large multilingual corpora.
 - General-purpose language knowledge, such as grammar, syntax, and semantics, is provided by these models.
 - Sentence embeddings are produced for general semantic tasks.
 Specific knowledge about distinguishing fake vs. real news is not included.

### Adaptation for Fake News Detection
- A small classification layer is added on top of the pretrained model.

- This layer is designed to map embeddings to target labels: fake or real.

### Fine-tuning Process
- The combined model (pretrained base + classifier) is fine-tuned on a labeled dataset of fake and real news.
- During fine-tuning, the weights of the pretrained model are adjusted to capture patterns indicative of fake or real news.
- The classifier layer is trained to map these patterns to the correct labels.

### Task-Specific Capability

- After fine-tuning, task-specific embeddings are produced.

- The model is enhanced to recognize patterns relevant to fake news detection.

Consequently, news articles can be classified as fake or real using the fine-tuned model.

In [ ]:
from transformers import  BertModel, Trainer, TrainingArguments
import torch
import torch.nn as nn
from classifier import CustomMLP
# 1️⃣ Load tokenizer and base transformer
# tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/bert-base-multilingual-uncased")
from transformers import BertTokenizer, TFBertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
base_model = BertModel.from_pretrained("bert-base-multilingual-uncased")

# Example: texts and labels lists
# texts = ["some news text", "another news"]
# labels = [0, 1]  # 0 = Real, 1 = Fake

# Train/test split
texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Tokenize
enc_train = tokenizer(texts_train, truncation=True, padding=True, max_length=512, return_tensors="pt")
enc_test  = tokenizer(texts_test, truncation=True, padding=True, max_length=512, return_tensors="pt")

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({
    "input_ids": enc_train["input_ids"],
    "attention_mask": enc_train["attention_mask"],
    "labels": torch.tensor(labels_train)
})

test_dataset = Dataset.from_dict({
    "input_ids": enc_test["input_ids"],
    "attention_mask": enc_test["attention_mask"],
    "labels": torch.tensor(labels_test)
})



# Instantiate the model
model = CustomMLP(base_model=base_model, num_labels=2)


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_strategy="epoch",
    # evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    learning_rate=5e-5,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)

trainer.train()


In [ ]:
# -----------------------------
# 7️⃣ Save model and tokenizer
# -----------------------------
tokenizer.save_pretrained("./fine_tuned_tokenizer_bert_base_multilingual_uncased")
torch.save(model.state_dict(), "./custom_mlp_bert_base_multilingual_uncased.pt")

In [ ]:
# 1️⃣ Reload tokenizer and model
tokenizer = BertTokenizer.from_pretrained("./fine_tuned_tokenizer_bert_base_multilingual_uncased")
model = CustomMLP(base_model=base_model, num_labels=2)
model.load_state_dict(torch.load("./custom_mlp_bert_base_multilingual_uncased.pt"))

# 2️⃣ Set device and move model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
def predict_news(text):
    text = re.sub(r'\s+', ' ', text).strip().lower()
    enc = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors="pt")
    
    # Move inputs to same device as model
    enc = {k: v.to(device) for k, v in enc.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model(**enc)
    logits = outputs["logits"]
    pred = torch.argmax(logits, dim=1).item()
    return "Fake" if pred == 1 else "Real"


# 3️⃣ Test loop
predictions = []
for text in texts_test:
    pred_label = predict_news(text)
    predictions.append(0 if pred_label=="Real" else 1)

accuracy = accuracy_score(labels_test, predictions)
print("Test Accuracy:", accuracy)
print(classification_report(labels_test, predictions, target_names=['Real', 'Fake']))
cm = confusion_matrix(labels_test, predictions)
print("Confusion Matrix:\n", cm)


Test Accuracy: 0.9997918690162342
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00     10000
        Fake       1.00      1.00      1.00      4414

    accuracy                           1.00     14414
   macro avg       1.00      1.00      1.00     14414
weighted avg       1.00      1.00      1.00     14414

Confusion Matrix:
 [[9998    2]
 [   1 4413]]


### Finetunning xlm-roberta-base transformer model for deteting fake news or real news

In [ ]:
from transformers import  AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from classifier import CustomMLP
# 1️⃣ Load tokenizer and base transformer
# tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/bert-base-multilingual-uncased")
tokenizer_roberta = AutoTokenizer.from_pretrained("xlm-roberta-base")
base_model_roberta = AutoModel.from_pretrained("xlm-roberta-base")

# Example: texts and labels lists
# texts = ["some news text", "another news"]
# labels = [0, 1]  # 0 = Real, 1 = Fake

# Train/test split
texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Tokenize
enc_train_roberta = tokenizer_roberta(texts_train, truncation=True, padding=True, max_length=512, return_tensors="pt")
enc_test_roberta = tokenizer_roberta(texts_test, truncation=True, padding=True, max_length=512, return_tensors="pt")

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({
    "input_ids": enc_train_roberta["input_ids"],
    "attention_mask": enc_train_roberta["attention_mask"],
    "labels": torch.tensor(labels_train)
})

test_dataset = Dataset.from_dict({
    "input_ids": enc_test_roberta["input_ids"],
    "attention_mask": enc_test_roberta["attention_mask"],
    "labels": torch.tensor(labels_test)
})



# Instantiate the model
model_roberta = CustomMLP(base_model=base_model_roberta, num_labels=2)


training_args = TrainingArguments(
    output_dir="./results_roberta",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_strategy="epoch",
    # evaluation_strategy="epoch",
    logging_dir="./logs_roberta",
    logging_steps=50,
    learning_rate=5e-5,
    save_total_limit=1
)

trainer = Trainer(
    model=model_roberta,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)

trainer.train()


Step,Training Loss
50,0.657500
100,0.590700
150,0.608200
200,0.593900
250,0.581200
300,0.539600
350,0.569200
400,0.608000
450,0.633000
500,0.551700


TrainOutput(global_step=3604, training_loss=0.26666524191567886, metrics={'train_runtime': 456.5406, 'train_samples_per_second': 126.282, 'train_steps_per_second': 7.894, 'total_flos': 0.0, 'train_loss': 0.26666524191567886, 'epoch': 1.0})

In [ ]:
# -----------------------------
# 7️⃣ Save model and tokenizer
# -----------------------------
tokenizer_roberta.save_pretrained("./fine_tuned_tokenizer_xlm_roberta_base")
torch.save(model_roberta.state_dict(), "./custom_mlp_xlm_roberta_base.pt")

In [ ]:
# 1️⃣ Reload tokenizer and model
tokenizer_roberta = AutoTokenizer.from_pretrained("./fine_tuned_tokenizer_xlm_roberta_base")
model_roberta = CustomMLP(base_model=base_model_roberta, num_labels=2)
model_roberta.load_state_dict(torch.load("./custom_mlp_xlm_roberta_base.pt"))

# 2️⃣ Set device and move model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_roberta.to(device)
def predict_news(text):
    text = re.sub(r'\s+', ' ', text).strip().lower()
    enc = tokenizer_roberta(text, truncation=True, padding=True, max_length=512, return_tensors="pt")
    
    # Move inputs to same device as model
    enc = {k: v.to(device) for k, v in enc.items()}

    model_roberta.eval()
    with torch.no_grad():
        outputs = model_roberta(**enc)
    logits = outputs["logits"]
    pred = torch.argmax(logits, dim=1).item()
    return "Fake" if pred == 1 else "Real"


# 3️⃣ Test loop
predictions = []
for text in texts_test:
    pred_label = predict_news(text)
    predictions.append(0 if pred_label=="Real" else 1)

accuracy = accuracy_score(labels_test, predictions)
print("Test Accuracy:", accuracy)
print(classification_report(labels_test, predictions, target_names=['Real', 'Fake']))
cm = confusion_matrix(labels_test, predictions)
print("Confusion Matrix:\n", cm)


Test Accuracy: 0.9941029554599695
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00     10000
        Fake       0.99      0.99      0.99      4414

    accuracy                           0.99     14414
   macro avg       0.99      0.99      0.99     14414
weighted avg       0.99      0.99      0.99     14414

Confusion Matrix:
 [[9961   39]
 [  46 4368]]


In [ ]:
check_data=pd.read_csv("final_dataset.csv",encoding="utf-8")